In [ ]:
import os
import re

def rename_query_camera_to_c1(folder_path):
    # 确保文件夹存在
    if not os.path.exists(folder_path):
        print(f"错误: 找不到文件夹 '{folder_path}'")
        return

    # 正则表达式解释：
    # ^(.+)_c    : 捕获组1：匹配开头到 '_c' 之前的所有内容
    # \d+        : 匹配紧跟在 '_c' 后的一个或多个数字（即我们要修改的部分）
    # (s.+)      : 捕获组2：匹配从 's' 开始到结尾的所有内容（s表示sequence）
    pattern = re.compile(r"^(.+)_c\d+(s.+)")

    count = 0
    print(f"正在处理文件夹: {folder_path} ...")
    
    for filename in os.listdir(folder_path):
        match = pattern.match(filename)
        
        if match:
            # 拼接新文件名：[ID部分]_c1[后续序列和后缀部分]
            prefix = match.group(1)
            suffix = match.group(2)
            new_filename = f"{prefix}_c1{suffix}"
            
            # 如果已经是 c1，则跳过
            if filename == new_filename:
                continue
            
            old_path = os.path.join(folder_path, filename)
            new_path = os.path.join(folder_path, new_filename)
            
            try:
                os.rename(old_path, new_path)
                print(f"重命名: {filename} -> {new_filename}")
                count += 1
            except Exception as e:
                print(f"处理 {filename} 时出错: {e}")

    print(f"\n✨ 处理完成！共修改了 {count} 个文件的摄像头编号。")

# --- 执行区 ---
# 建议先在少量备份文件上测试
target_folder = '/home/xiaolei/projects/data/ReID/Qiuxiu/bounding_box_train'
rename_query_camera_to_c1(target_folder)

In [ ]:
import os
import shutil
from collections import Counter

def move_single_image_ids(src_path, dest_path):
    """
    将 Market-1501 路径下仅有一张图片的类别移动到目标文件夹
    """
    # 1. 检查并创建目标路径
    if not os.path.exists(dest_path):
        os.makedirs(dest_path)
        print(f"创建目标目录: {dest_path}")

    if not os.path.exists(src_path):
        print(f"错误: 找不到源路径 {src_path}")
        return

    # 2. 统计 ID 出现频率
    file_list = [f for f in os.listdir(src_path) if f.endswith(('.jpg', '.png'))]
    
    # 建立 文件名 -> ID 的映射，方便后续根据 ID 找文件
    file_to_id = {}
    ids = []
    for f in file_list:
        pid = f.split('_')[0]
        if pid not in ['-1', '0000']:
            ids.append(pid)
            file_to_id[f] = pid

    id_counts = Counter(ids)

    # 3. 找出只有一张图片的 ID
    single_image_ids = {pid for pid, count in id_counts.items() if count == 1}
    
    print(f"总 ID 数: {len(id_counts)}")
    print(f"发现只有一张图片的 ID 数量: {len(single_image_ids)}")
    print("-" * 30)

    # 4. 执行移动操作
    move_count = 0
    for filename, pid in file_to_id.items():
        if pid in single_image_ids:
            src_file = os.path.join(src_path, filename)
            dest_file = os.path.join(dest_path, filename)
            
            # 移动文件
            shutil.move(src_file, dest_file)
            print(f"已移动: {filename} (ID: {pid})")
            move_count += 1

    print("-" * 30)
    print(f"操作完成！共移动了 {move_count} 张图片到 {dest_path}")

# --- 执行区 ---
# 源数据集路径
src_dataset = '/home/xiaolei/projects/data/ReID/QiuxiuOri/bounding_box_train'
# 移动到的目标路径
trash_path = '/home/xiaolei/projects/data/ReID/QiuxiuOri/single_images_backup'

move_single_image_ids(src_dataset, trash_path)

In [ ]:
import os
import shutil
from collections import defaultdict

def rename_market_ids_to_continuous(root_path, save_path=None):
    """
    将 Market-1501 格式的 ID 重命名为从 1 开始的连续 ID
    :param root_path: 包含 query 和 bounding_box_test 的父目录
    :param save_path: 保存新数据的路径（若为 None 则在原文件夹修改，建议备份）
    """
    query_dir = os.path.join(root_path, 'query')
    test_dir = os.path.join(root_path, 'bounding_box_test')
    
    if not os.path.exists(query_dir) or not os.path.exists(test_dir):
        print("错误: 找不到 query 或 bounding_box_test 文件夹")
        return

    # 1. 搜集两个目录下所有的原始 PID
    def get_all_pids(dirs):
        all_pids = set()
        for d in dirs:
            files = [f for f in os.listdir(d) if f.endswith(('.jpg', '.png'))]
            for f in files:
                pid = f.split('_')[0]
                # 排除干扰项
                if pid not in ['-1', '0000']:
                    all_pids.add(pid)
        return sorted(list(all_pids))

    original_pids = get_all_pids([query_dir, test_dir])
    
    # 2. 建立 ID 映射表 (例如: {'0002': '0001', '0005': '0002', ...})
    # 这里映射为 4 位字符串，符合 Market 格式规范
    pid_map = {old: f"{i+1:06d}" for i, old in enumerate(original_pids)}
    
    print(f"统计完成：共有 {len(original_pids)} 个有效 ID")
    print(f"映射示例: {list(pid_map.items())[:3]}")

    # 3. 执行文件处理
    sub_sets = ['query', 'bounding_box_test']
    
    for sub in sub_sets:
        src_sub_dir = os.path.join(root_path, sub)
        # 如果指定了 save_path，则创建新目录，否则直接修改原目录
        dst_sub_dir = os.path.join(save_path, sub) if save_path else src_sub_dir
        
        if not os.path.exists(dst_sub_dir):
            os.makedirs(dst_sub_dir)

        files = [f for f in os.listdir(src_sub_dir) if f.endswith(('.jpg', '.png'))]
        
        for f in files:
            parts = f.split('_')
            old_pid = parts[0]
            
            # 如果是有效 ID，则重命名；如果是 -1 或 0000，保持不变
            if old_pid in pid_map:
                new_pid = pid_map[old_pid]
                new_filename = "_".join([new_pid] + parts[1:])
            else:
                new_filename = f # 保持原样 (-1, 0000)

            src_file = os.path.join(src_sub_dir, f)
            dst_file = os.path.join(dst_sub_dir, new_filename)

            if save_path:
                shutil.copy(src_file, dst_file)
            else:
                os.rename(src_file, dst_file)

    print(f"处理完成！新数据集已保存至: {save_path if save_path else root_path}")

# --- 执行区 ---
# 建议先备份原始数据，或者指定一个新的 save_path
dataset_root = '/home/xiaolei/projects/data/ReID/Qiuxiu_false'
new_dataset_root = '/home/xiaolei/projects/data/ReID/Qiuxiu'

rename_market_ids_to_continuous(dataset_root, new_dataset_root)

In [ ]:
import os
import shutil
from collections import Counter

def rename_train_only(root_path, save_path):
    """
    仅重命名 bounding_box_train 目录下的 ID
    """
    train_dir = os.path.join(root_path, 'bounding_box_train')
    dest_dir = os.path.join(save_path, 'bounding_box_train')

    if not os.path.exists(train_dir):
        print(f"错误: 找不到训练集路径 {train_dir}")
        return

    # 1. 扫描所有文件并提取唯一 ID
    print("正在统计训练集 ID...")
    files = [f for f in os.listdir(train_dir) if f.endswith(('.jpg', '.png'))]
    
    unique_pids = set()
    for f in files:
        pid = f.split('_')[0]
        if pid not in ['-1', '0000']:
            unique_pids.add(pid)
    
    # 2. 建立映射表 (从 0001 开始)
    sorted_pids = sorted(list(unique_pids))
    pid_map = {old: f"{i+1:06d}" for i, old in enumerate(sorted_pids)}
    
    print(f"统计完成：共有 {len(sorted_pids)} 个训练类别 (Person IDs)")
    print(f"新 ID 范围: 000001 - {len(sorted_pids):06d}")

    # 3. 执行拷贝与重命名
    if not os.path.exists(dest_dir):
        os.makedirs(dest_dir)
        print(f"创建目标目录: {dest_dir}")

    for f in files:
        parts = f.split('_')
        old_pid = parts[0]
        
        if old_pid in pid_map:
            new_pid = pid_map[old_pid]
            # 拼接新文件名：新ID_原名剩余部分
            new_filename = "_".join([new_pid] + parts[1:])
        else:
            # -1 或 0000 保持不变
            new_filename = f
            
        shutil.copy(os.path.join(train_dir, f), os.path.join(dest_dir, new_filename))

    print(f"成功！处理后的文件已存入: {dest_dir}")

# --- 执行区 ---
raw_root = '/home/xiaolei/projects/data/ReID/Qiuxiu_false'
new_root = '/home/xiaolei/projects/data/ReID/Qiuxiu_continuous'

rename_train_only(raw_root, new_root)

In [ ]:
import os
import random
import matplotlib.pyplot as plt
from PIL import Image

def visualize_reid_mapping(root_path, num_samples=50):
    query_dir = os.path.join(root_path, 'query')
    test_dir = os.path.join(root_path, 'bounding_box_test')

    # 1. 获取所有 ID 及其对应的文件列表
    def get_id_dict(path):
        id_dict = {}
        files = [f for f in os.listdir(path) if f.endswith(('.jpg', '.png'))]
        for f in files:
            pid = f.split('_')[0]
            if pid not in ['-1', '0000']:
                if pid not in id_dict:
                    id_dict[pid] = []
                id_dict[pid].append(f)
        return id_dict

    query_data = get_id_dict(query_dir)
    test_data = get_id_dict(test_dir)

    # 2. 找出两个集合共同拥有的 ID
    common_pids = list(set(query_data.keys()) & set(test_data.keys()))
    
    if not common_pids:
        print("错误：未在两个目录中找到共同的 ID，请检查路径或重命名逻辑。")
        return

    # 随机抽取指定数量的 ID
    sampled_pids = random.sample(common_pids, min(num_samples, len(common_pids)))
    print(f"随机抽取了 {len(sampled_pids)} 个 ID 进行比对展示...")

    # 3. 开始绘图 (每行展示 5 个 ID，每个 ID 占两列，总共 10 列)
    cols = 10 
    rows = (len(sampled_pids) + 4) // 5  # 计算需要的行数
    
    plt.figure(figsize=(20, 4 * rows))
    plt.suptitle("Re-ID Mapping Verification (Left: Query | Right: Test)", fontsize=20)

    for i, pid in enumerate(sampled_pids):
        # 随机选一张 Query 和一张 Test 图片
        q_img_name = random.choice(query_data[pid])
        t_img_name = random.choice(test_data[pid])

        # 加载图片
        q_img = Image.open(os.path.join(query_dir, q_img_name))
        t_img = Image.open(os.path.join(test_dir, t_img_name))

        # 绘制 Query 图 (左)
        ax_q = plt.subplot(rows, cols, 2 * i + 1)
        ax_q.imshow(q_img)
        ax_q.set_title(f"ID:{pid} (Q)", color='blue')
        ax_q.axis('off')

        # 绘制 Test 图 (右)
        ax_t = plt.subplot(rows, cols, 2 * i + 2)
        ax_t.imshow(t_img)
        ax_t.set_title(f"ID:{pid} (T)", color='green')
        ax_t.axis('off')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

# --- 执行区 ---
# 填入你重命名后的数据集根目录
dataset_root = '/home/xiaolei/projects/data/ReID/Qiuxiu'
visualize_reid_mapping(dataset_root, num_samples=50)

In [ ]:
import os
from collections import Counter

def count_market1501_categories(data_path):
    """
    统计 Market-1501 数据集中每个类别的图片数量
    :param data_path: 数据集文件夹路径 (如 'bounding_box_train')
    """
    if not os.path.exists(data_path):
        print(f"错误: 找不到路径 {data_path}")
        return

    # 获取目录下所有以 .jpg 或 .png 结尾的文件
    file_list = [f for f in os.listdir(data_path) if f.endswith(('.jpg', '.png'))]
    
    # 提取所有 ID
    # 文件名格式: 0002_c1s1_000456_03.jpg -> 提取 '0002'
    ids = []
    for filename in file_list:
        pid = filename.split('_')[0]
        # 排除干扰项（如 -1 或 0000 通常代表背景或干扰）
        if pid not in ['-1', '0000']:
            ids.append(pid)

    # 使用 Counter 进行统计
    id_counts = Counter(ids)
    
    # 排序：按 ID 数字大小排序
    sorted_ids = sorted(id_counts.keys(), key=lambda x: int(x))

    print(f"统计路径: {data_path}")
    print(f"总图片数: {len(ids)}")
    print(f"总类别数 (Person IDs): {len(sorted_ids)}")
    print("-" * 30)
    print(f"{'ID':<10} | {'图片数量':<10}")
    print("-" * 30)
    
    for pid in sorted_ids:
        print(f"{pid:<10} | {id_counts[pid]:<10}")
    
    # 可选：打印统计摘要
    counts = list(id_counts.values())
    print("-" * 30)
    print(f"单类最多图片数: {max(counts)}")
    print(f"单类最少图片数: {min(counts)}")
    print(f"平均每类图片数: {sum(counts)/len(counts):.2f}")

# --- 执行区 ---
# 替换为你的 Market-1501 实际路径
dataset_path = "/home/xiaolei/projects/data/ReID/Qiuxiu/bounding_box_train"
count_market1501_categories(dataset_path)

In [ ]:
import os
import shutil
from collections import Counter

def move_single_image_ids(src_path, dest_path):
    """
    将 Market-1501 路径下仅有一张图片的类别移动到目标文件夹
    """
    # 1. 检查并创建目标路径
    if not os.path.exists(dest_path):
        os.makedirs(dest_path)
        print(f"创建目标目录: {dest_path}")

    if not os.path.exists(src_path):
        print(f"错误: 找不到源路径 {src_path}")
        return

    # 2. 统计 ID 出现频率
    file_list = [f for f in os.listdir(src_path) if f.endswith(('.jpg', '.png'))]
    
    # 建立 文件名 -> ID 的映射，方便后续根据 ID 找文件
    file_to_id = {}
    ids = []
    for f in file_list:
        pid = f.split('_')[0]
        if pid not in ['-1', '0000']:
            ids.append(pid)
            file_to_id[f] = pid

    id_counts = Counter(ids)

    # 3. 找出只有一张图片的 ID
    single_image_ids = {pid for pid, count in id_counts.items() if count == 1}
    
    print(f"总 ID 数: {len(id_counts)}")
    print(f"发现只有一张图片的 ID 数量: {len(single_image_ids)}")
    print("-" * 30)

    # 4. 执行移动操作
    move_count = 0
    for filename, pid in file_to_id.items():
        if pid in single_image_ids:
            src_file = os.path.join(src_path, filename)
            dest_file = os.path.join(dest_path, filename)
            
            # 移动文件
            shutil.move(src_file, dest_file)
            print(f"已移动: {filename} (ID: {pid})")
            move_count += 1

    print("-" * 30)
    print(f"操作完成！共移动了 {move_count} 张图片到 {dest_path}")

# --- 执行区 ---
# 源数据集路径
src_dataset = '/home/xiaolei/projects/data/ReID/VIPerson/bounding_box_train'
# 移动到的目标路径
trash_path = '/home/xiaolei/projects/data/ReID/VIPerson/single_images_backup'

move_single_image_ids(src_dataset, trash_path)

In [ ]:
import os
from collections import defaultdict

# 定义数据集核心配置
DATASET_NAME = "Qiuxiu"
FOLDER_NAMES = ["bounding_box_test", "bounding_box_train", "query"]
SUPPORTED_IMAGE_FORMATS = ('.jpg', '.jpeg', '.png', '.bmp', '.gif')

def parse_image_filename(filename):
    """解析图片文件名，提取球员ID和图片序号"""
    try:
        file_base = os.path.splitext(filename)[0]
        parts = file_base.split('_')
        if len(parts) >= 3:
            player_id = parts[0]
            image_idx = parts[2]
            if player_id.isdigit() and image_idx.isdigit():
                return player_id, image_idx
        return None, None
    except:
        return None, None

def stat_single_folder(folder_path):
    """统计单个文件夹的数据集分布（新增球员图片数列表，用于计算平均）"""
    image_total_count = 0
    player_image_dict = defaultdict(list)
    
    for filename in os.listdir(folder_path):
        if filename.startswith('.') or not filename.lower().endswith(SUPPORTED_IMAGE_FORMATS):
            continue
        
        player_id, image_idx = parse_image_filename(filename)
        if player_id is not None and image_idx is not None:
            image_total_count += 1
            player_image_dict[player_id].append(image_idx)
    
    sorted_player_ids = sorted(player_image_dict.keys())
    # 新增：提取每个球员的图片数量列表，用于后续计算平均
    player_image_counts = [len(img_list) for img_list in player_image_dict.values()]
    stat_result = {
        "image_total": image_total_count,
        "player_dict": {pid: len(img_list) for pid, img_list in player_image_dict.items()},
        "player_count": len(sorted_player_ids),
        "player_image_counts": player_image_counts  # 新增字段
    }
    return stat_result

def stat_qiuxiu_dataset(dataset_root):
    """统计整个数据集分布，增加平均图片数输出"""
    print("=" * 80)
    print(f"开始统计 {DATASET_NAME} ReID 数据集分布（含平均图片数）")
    print(f"数据集根目录：{dataset_root}")
    print("=" * 80 + "\n")
    
    if not os.path.exists(dataset_root):
        raise FileNotFoundError(f"数据集根目录不存在：{dataset_root}")
    if not os.path.isdir(dataset_root):
        raise NotADirectoryError(f"指定路径不是文件夹：{dataset_root}")
    
    # 全局统计容器：累加所有图片，合并所有球员的图片数（去重ID）
    total_overall_images = 0
    global_player_image_dict = defaultdict(int)  # key: 球员ID, value: 全局总图片数
    existing_folders = []
    
    for folder_name in FOLDER_NAMES:
        print(f'folder_name: {folder_name}')
        folder_path = os.path.join(dataset_root, folder_name)
        if not os.path.exists(folder_path) or not os.path.isdir(folder_path):
            print(f"⚠️  警告：文件夹 {folder_name} 不存在或非文件夹，跳过统计")
            continue
        
        existing_folders.append(folder_name)
        print(f"---------- 统计文件夹：{folder_name} ----------")
        stat_result = stat_single_folder(folder_path)
        
        # 提取单个文件夹统计结果
        img_total = stat_result["image_total"]
        player_count = stat_result["player_count"]
        player_dict = stat_result["player_dict"]
        player_image_counts = stat_result["player_image_counts"]
        
        # 计算并输出单个文件夹的平均图片数
        if player_count > 0:
            folder_avg_images = img_total / player_count
        else:
            folder_avg_images = 0.0
        
        # 输出文件夹级统计（新增平均图片数）
        print(f"1. 图片总数量：{img_total}")
        print(f"2. 唯一球员ID数量：{player_count}")
        print(f"3. 该文件夹球员平均图片数：{folder_avg_images:.2f} 张（保留2位小数）")
        print(f"4. 各球员图片数量分布（前10个ID）：")
        
        sorted_player_items = sorted(player_dict.items())
        for i, (player_id, img_count) in enumerate(sorted_player_items[:10]):
            print(f"   - 球员ID {player_id}：{img_count} 张图片")
        if len(sorted_player_items) > 10:
            print(f"   - ... 还有 {len(sorted_player_items) - 10} 个球员ID未展示")
        
        # 累加全局统计（合并球员ID，累加图片数）
        total_overall_images += img_total
        for player_id, img_count in player_dict.items():
            global_player_image_dict[player_id] += img_count
        print("\n")
    
    # 计算并输出全局平均图片数
    global_player_count = len(global_player_image_dict)
    if global_player_count > 0:
        global_avg_images = total_overall_images / global_player_count
    else:
        global_avg_images = 0.0
    
    # 全局汇总（新增全局平均图片数）
    print("=" * 80)
    print(f"{DATASET_NAME} 数据集全局汇总")
    print("=" * 80)
    print(f"1. 数据集总图片数量：{total_overall_images}")
    print(f"2. 数据集总唯一球员ID数量：{global_player_count}")
    print(f"3. 数据集全局球员平均图片数：{global_avg_images:.2f} 张（保留2位小数）")
    print(f"4. 包含文件夹：{existing_folders}")
    print("\n数据集统计完成！")

# ------------------- 配置参数（修改为你的数据集根目录路径） -------------------
if __name__ == "__main__":
    # 数据集根目录：包含 bounding_box_test、bounding_box_train、query 三个文件夹
    DATASET_ROOT = "/home/xiaolei/projects/data/ReID/Qiuxiu"  # 替换为你的实际数据集根目录路径
    
    try:
        stat_qiuxiu_dataset(DATASET_ROOT)
    except Exception as e:
        print(f"❌ 统计过程中出错：{e}")